# Predicción del cambio horario de analitos en cristales con RidgeCV

Notebook para estimar por separado analitos de cristales sin refinar y refinados. El objetivo es delta(t→t+1 h) = valor(t+1 h) − valor(t). La evaluación usa una partición cronológica 80/20.

La hoja de datos ya tiene frecuencia horaria. Se rellenan hacia delante huecos de hasta 2 horas en predictores y el resto se imputa con la mediana aprendida solo en entrenamiento. StandardScaler y RidgeCV están en el pipeline.

## 1. Configuración

Dependencias: pandas, numpy, scikit-learn y openpyxl. Cambia RUTA_ARCHIVO si el Excel está en otra ubicación.

In [1]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RUTA_EXCEL_ESCRITORIO = Path("C:/Users/SrLeo/Desktop/SQM/Predicción efecto de analitos/tabla_limpia_valores.xlsx")
RUTA_EXCEL_RELATIVA = Path("tabla_limpia_valores.xlsx")
RUTA_ARCHIVO = next((p.resolve() for p in (RUTA_EXCEL_RELATIVA, RUTA_EXCEL_ESCRITORIO) if p.exists()), RUTA_EXCEL_ESCRITORIO)
HOJA_DATOS = "Sheet1"
COLUMNA_TIEMPO = "Fecha_hora"
HORIZONTE_HORAS = 2
FRACCION_ENTRENAMIENTO = 0.80
ALPHAS = np.logspace(-2, 5, 25)  # rango ampliado porque la búsqueda anterior llegaba a 1000

if not RUTA_ARCHIVO.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {RUTA_ARCHIVO}")

## 2. Carga y preparación temporal

La detección de nombres tolera tildes y la errata refinal del encabezado ClO4. Solo se leen los datos de Sheet1; las hojas de trazabilidad, incidencias y resumen no se usan como variables.

In [2]:
def normalizar_nombre(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", texto).strip().lower()

raw = pd.read_excel(RUTA_ARCHIVO, sheet_name=HOJA_DATOS)
if COLUMNA_TIEMPO not in raw.columns:
    raise KeyError(f"No existe {COLUMNA_TIEMPO!r}. Columnas: {list(raw.columns)}")

raw[COLUMNA_TIEMPO] = pd.to_datetime(raw[COLUMNA_TIEMPO], errors="coerce")
raw = raw.dropna(subset=[COLUMNA_TIEMPO]).sort_values(COLUMNA_TIEMPO)
if raw[COLUMNA_TIEMPO].duplicated().any():
    raise ValueError("Hay timestamps duplicados; resuélvelos antes de modelar.")

data = raw.set_index(COLUMNA_TIEMPO).sort_index()
data = data.asfreq("h")  # crea horas faltantes con valores vacíos; no inventa mediciones
numeric = data.select_dtypes(include="number").copy()

columnas_normalizadas = {col: normalizar_nombre(col) for col in numeric.columns}
objetivos_sin_refinar = [c for c, n in columnas_normalizadas.items() if n.startswith("cristal sin refin")]
objetivos_refinado = [c for c, n in columnas_normalizadas.items() if n.startswith("cristal refin")]
if not objetivos_sin_refinar or not objetivos_refinado:
    raise ValueError("No se detectaron ambos grupos de objetivos; revisa encabezados.")

# Predictores químicos de ML Refi, SSK y Purga/Recirculación.
# Se excluyen analitos de cristal para evitar usar los objetivos como predictores.
prefijos_quimicos = ("ml_refi", "ssk", "purga_recirculacion", "purga recirculacion")
features = [
    c for c, n in columnas_normalizadas.items()
    if n.startswith(prefijos_quimicos) and c not in objetivos_sin_refinar + objetivos_refinado
]
if not features:
    raise ValueError("No se encontraron predictores con los prefijos químicos esperados.")

print(f"Periodo: {data.index.min()} a {data.index.max()}")
print(f"Filas horarias: {len(data):,}")
print(f"Predictores químicos ({len(features)}): {features}")
print(f"Objetivos sin refinar ({len(objetivos_sin_refinar)}): {objetivos_sin_refinar}")
print(f"Objetivos refinados ({len(objetivos_refinado)}): {objetivos_refinado}")

# Relleno temporal solo hacia delante y solo hasta 2 horas.
X_todos = numeric[features].ffill(limit=2)
y_nivel = numeric[objetivos_sin_refinar + objetivos_refinado]
delta_futuro = y_nivel.shift(-HORIZONTE_HORAS) - y_nivel
delta_futuro.columns = [f"delta__{c}" for c in delta_futuro.columns]

display(delta_futuro.notna().sum().rename("deltas_disponibles").to_frame())

Periodo: 2025-01-01 01:00:00 a 2025-12-16 04:00:00
Filas horarias: 8,380
Predictores químicos (22): ['ML_REFI_NANO3', 'ML_REFI_SO4', 'ML_REFI_K', 'ML_REFI_CA', 'ML_REFI_MG', 'ML_REFI_KCLO4', 'ML_REFI_CLO4IC', 'SSK NANO3', 'SSK CL', 'SSK SO4', 'SSK K', 'SSK MG', 'SSK CA', 'SSK KCLO4 IC', 'SSKCLO4 IC', 'Purga_Recirculacion NaNO3', 'Purga_Recirculación SO4', 'Purga_Recirculacion K', 'Purga_Recirculación Ca', 'Purga_Recirculación Mg', 'Purga_Recirculación KClO4 IC', 'Purga_Recirculación ClO4 IC']
Objetivos sin refinar (5): ['Cristal sin refinar Cl', 'Cristal sin refinar SO4', 'Cristal sin refinar Ca', 'Cristal sin refinar Mg', 'Cristal sin refinal ClO4']
Objetivos refinados (5): ['Cristal Refinado Cl', 'Cristal Refinado SO4', 'Cristal Refinado Ca', 'Cristal Refinado Mg', 'Cristal refinado ClO4']


,deltas_disponibles
delta__Cristal sin refinar Cl,3849
delta__Cristal sin refinar SO4,3855
delta__Cristal sin refinar Ca,3511
delta__Cristal sin refinar Mg,3848
delta__Cristal sin refinal ClO4,3789
delta__Cristal Refinado Cl,3883
delta__Cristal Refinado SO4,3883
delta__Cristal Refinado Ca,3883
delta__Cristal Refinado Mg,3883
delta__Cristal refinado ClO4,3883


## 3. Entrenamiento y evaluación cronológica

Se ajusta un modelo por analito para aprovechar sus filas etiquetadas disponibles. El baseline predice delta igual a cero, equivalente a repetir el último valor observado.

In [3]:
resultados = []
predicciones = []
coeficientes = []
modelos = {}

# Un único corte de calendario para todos los objetivos.
# Se purgan del entrenamiento las etiquetas cuyo horizonte cruza el corte.
corte_temporal = data.index[int(len(data) * FRACCION_ENTRENAMIENTO)]
duracion_horizonte = pd.Timedelta(hours=HORIZONTE_HORAS)

def crear_folds_por_fecha(indice, n_splits=5, horizonte=duracion_horizonte):
    """Folds expansivos con bloques de validación de igual duración y purga del horizonte."""
    indice = pd.DatetimeIndex(indice)
    bordes = pd.date_range(
        start=indice.min(),
        end=indice.max() + pd.Timedelta(hours=1),
        periods=n_splits + 2
    )
    folds = []
    for k in range(n_splits):
        inicio_validacion = bordes[k + 1]
        fin_validacion = bordes[k + 2]
        mascara_train = (indice + horizonte) <= inicio_validacion
        mascara_validacion = (indice >= inicio_validacion) & (indice < fin_validacion)
        idx_train = np.flatnonzero(mascara_train)
        idx_validacion = np.flatnonzero(mascara_validacion)
        if len(idx_train) and len(idx_validacion):
            folds.append((idx_train, idx_validacion))
    if len(folds) < 2:
        raise ValueError("No se pudieron formar suficientes folds temporales por fecha.")
    return folds

for grupo, targets_grupo in [("Sin refinar", objetivos_sin_refinar), ("Refinado", objetivos_refinado)]:
    for target in targets_grupo:
        y_completo = delta_futuro[f"delta__{target}"]
        validos = y_completo.notna()
        X_validos = X_todos.loc[validos]
        y_validos = y_completo.loc[validos]

        # Entrenamiento antes del corte; cada etiqueta debe terminar como máximo en el corte.
        mascara_train = (X_validos.index + duracion_horizonte) <= corte_temporal
        mascara_test = X_validos.index >= corte_temporal
        X_train, y_train = X_validos.loc[mascara_train], y_validos.loc[mascara_train]
        X_test, y_test = X_validos.loc[mascara_test], y_validos.loc[mascara_test]

        if len(y_train) < 20 or len(y_test) == 0:
            print(f"Se omite {target}: datos insuficientes antes o después del corte.")
            continue

        cv_folds = crear_folds_por_fecha(X_train.index)
        pipeline = make_pipeline(
            SimpleImputer(strategy="median", add_indicator=False),
            StandardScaler(),
            Ridge()
        )
        busqueda = GridSearchCV(
            estimator=pipeline,
            param_grid={"ridge__alpha": ALPHAS},
            scoring="neg_mean_absolute_error",
            cv=cv_folds,
            refit=True,
            n_jobs=-1
        )
        busqueda.fit(X_train, y_train)
        modelo = busqueda.best_estimator_
        pred = modelo.predict(X_test)

        mae_ridge = mean_absolute_error(y_test, pred)
        mae_baseline = mean_absolute_error(y_test, np.zeros_like(y_test))
        resultados.append({
            "grupo": grupo, "objetivo": target,
            "fecha_inicio_test": X_test.index.min(),
            "fecha_fin_test": X_test.index.max(),
            "n_train": len(X_train), "n_test": len(X_test),
            "alpha_elegido": busqueda.best_params_["ridge__alpha"],
            "MAE_CV": -busqueda.best_score_,
            "MAE_Ridge": mae_ridge,
            "MAE_baseline_delta_0": mae_baseline,
            "mejora_MAE_vs_baseline": mae_baseline - mae_ridge
        })
        modelos[target] = modelo

        predicciones.append(pd.DataFrame({
            "grupo": grupo, "objetivo": target,
            "delta_real": y_test, "delta_predicho": pred,
            "nivel_actual": y_nivel.loc[y_test.index, target],
            "nivel_futuro_real": y_nivel[target].shift(-HORIZONTE_HORAS).loc[y_test.index],
            "nivel_futuro_predicho": y_nivel.loc[y_test.index, target] + pred
        }, index=y_test.index).rename_axis("timestamp_t"))

        nombres_transformados = modelo.named_steps["simpleimputer"].get_feature_names_out(features)
        coeficientes.append(pd.DataFrame({
            "grupo": grupo, "objetivo": target,
            "predictor_transformado": nombres_transformados,
            "coeficiente_estandarizado": modelo.named_steps["ridge"].coef_
        }))

if not resultados:
    raise RuntimeError("No se ajustaron modelos; revisa la cobertura de los objetivos y el horizonte.")

metricas = pd.DataFrame(resultados).sort_values(["grupo", "objetivo"]).reset_index(drop=True)
predicciones_test = pd.concat(predicciones).sort_index() if predicciones else pd.DataFrame()
coeficientes_modelos = pd.concat(coeficientes, ignore_index=True) if coeficientes else pd.DataFrame()

display(metricas)

,grupo,objetivo,fecha_inicio_test,fecha_fin_test,n_train,n_test,alpha_elegido,MAE_CV,MAE_Ridge,MAE_baseline_delta_0,mejora_MAE_vs_baseline
0,Refinado,Cristal Refinado Ca,2025-10-08 23:00:00,2025-12-15 23:00:00,3089,794,100000.000000,0.002052,0.002244,0.002229,-0.000014
1,Refinado,Cristal Refinado Cl,2025-10-08 23:00:00,2025-12-15 23:00:00,3089,794,100000.000000,0.017352,0.006097,0.005693,-0.000405
2,Refinado,Cristal Refinado Mg,2025-10-08 23:00:00,2025-12-15 23:00:00,3089,794,100000.000000,0.001613,0.001920,0.001917,-0.000003
3,Refinado,Cristal Refinado SO4,2025-10-08 23:00:00,2025-12-15 23:00:00,3089,794,100000.000000,0.014661,0.018347,0.018325,-0.000022
4,Refinado,Cristal refinado ClO4,2025-10-08 23:00:00,2025-12-15 23:00:00,3089,794,100000.000000,6.233591,6.099755,6.088161,-0.011594
5,Sin refinar,Cristal sin refinal ClO4,2025-10-08 19:00:00,2025-12-15 23:00:00,3050,739,13335.214322,83.459548,112.636566,111.958733,-0.677833
6,Sin refinar,Cristal sin refinar Ca,2025-10-08 19:00:00,2025-12-15 23:00:00,2717,794,100000.000000,0.003512,0.003096,0.003086,-0.000011
7,Sin refinar,Cristal sin refinar Cl,2025-10-08 19:00:00,2025-12-15 23:00:00,3055,794,26101.572157,0.120160,0.103030,0.102985,-0.000046
8,Sin refinar,Cristal sin refinar Mg,2025-10-08 19:00:00,2025-12-15 23:00:00,3057,791,13335.214322,0.013390,0.016192,0.016217,0.000026
9,Sin refinar,Cristal sin refinar SO4,2025-10-08 19:00:00,2025-12-15 23:00:00,3061,794,100000.000000,0.029395,0.034714,0.034710,-0.000003


## 4. Revisión de resultados

MAE Ridge menor que el baseline indica mejora en el conjunto de prueba para ese analito. El nivel predicho se reconstruye como valor(t) + delta_predicho.

In [4]:
display(predicciones_test.head(20))

coeficientes_ordenados = coeficientes_modelos.assign(
    magnitud=lambda tabla: tabla["coeficiente_estandarizado"].abs()
).sort_values(["objetivo", "magnitud"], ascending=[True, False])

display(coeficientes_ordenados.drop(columns="magnitud").groupby("objetivo", sort=False).head(310))

,grupo,objetivo,delta_real,delta_predicho,nivel_actual,nivel_futuro_real,nivel_futuro_predicho
timestamp_t,,,,,,,
2025-10-08 19:00:00,Sin refinar,Cristal sin refinar Cl,0.010,-0.003694,0.400,0.410,0.396306
2025-10-08 19:00:00,Sin refinar,Cristal sin refinar Ca,0.000,0.000056,0.030,0.030,0.030056
2025-10-08 19:00:00,Sin refinar,Cristal sin refinar Mg,-0.005,0.000268,0.059,0.054,0.059268
2025-10-08 19:00:00,Sin refinar,Cristal sin refinal ClO4,-8.000,4.528637,99.000,91.000,103.528637
2025-10-08 19:00:00,Sin refinar,Cristal sin refinar SO4,-0.010,0.000357,0.200,0.190,0.200357
2025-10-08 21:00:00,Sin refinar,Cristal sin refinar Ca,0.000,0.000049,0.030,0.030,0.030049
2025-10-08 21:00:00,Sin refinar,Cristal sin refinar Cl,0.050,-0.003278,0.410,0.460,0.406722
2025-10-08 21:00:00,Sin refinar,Cristal sin refinal ClO4,3.000,4.033085,91.000,94.000,95.033085
2025-10-08 21:00:00,Sin refinar,Cristal sin refinar SO4,0.000,0.000401,0.190,0.190,0.190401


,grupo,objetivo,predictor_transformado,coeficiente_estandarizado
163,Refinado,Cristal Refinado Ca,SSK SO4,0.000008
166,Refinado,Cristal Refinado Ca,SSK CA,0.000008
164,Refinado,Cristal Refinado Ca,SSK K,0.000007
154,Refinado,Cristal Refinado Ca,ML_REFI_NANO3,0.000005
170,Refinado,Cristal Refinado Ca,Purga_Recirculación SO4,0.000005
...,...,...,...,...
41,Sin refinar,Cristal sin refinar SO4,Purga_Recirculación Mg,-0.000006
42,Sin refinar,Cristal sin refinar SO4,Purga_Recirculación KClO4 IC,-0.000006
33,Sin refinar,Cristal sin refinar SO4,SSK MG,0.000005
43,Sin refinar,Cristal sin refinar SO4,Purga_Recirculación ClO4 IC,-0.000005


## 5. Guardar resultados (opcional)

Guarda métricas, predicciones de prueba y coeficientes en una subcarpeta junto al Excel.

In [5]:
CARPETA_SALIDA = RUTA_ARCHIVO.parent / "resultados_ridgecv_delta"
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
metricas.to_csv(CARPETA_SALIDA / "metricas_ridgecv.csv", index=False, encoding="utf-8-sig")
predicciones_test.to_csv(CARPETA_SALIDA / "predicciones_test_delta.csv", encoding="utf-8-sig")
coeficientes_modelos.to_csv(CARPETA_SALIDA / "coeficientes_estandarizados.csv", index=False, encoding="utf-8-sig")
print(f"Archivos guardados en: {CARPETA_SALIDA}")

Archivos guardados en: /mnt/c/Users/SrLeo/Desktop/SQM/Predicción efecto de analitos/resultados_ridgecv_delta
